# 참고용 사본 (Reference copy)

이 노트북은 원 연구 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 **참고용 사본**이다. 뉴스 원문, 일별·분봉 가격, 경매 거래대금 등 **공개하지 않은 데이터**(`news_tokenized_labeled*.pkl`, `daily_adjusted_close.parquet` 등)가 있어야 처음부터 끝까지 실행되며, 이 저장소만으로는 실행되지 않는다. 분석 구조와 코드를 읽는 용도다. 이 저장소에서 바로 실행되는 것은 `00_public_sample_workflow.ipynb`와 `../code/`뿐이다.

This is a reference copy of an original research notebook: outputs, execution counts, local paths and keys were removed. It needs non-public data (news text, daily/minute prices, auction values) and will not run from this repository alone.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# =============================================================================
# 1. 데이터 불러오기
# =============================================================================
path = "news_tokenized_labeled.pkl"
df = pd.read_pickle(path)

---
### **SESTM**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

TRAIN_YEARS = 5
N_POS_WORDS = 200
N_NEG_WORDS = 200
MIN_WORD_COUNT = 50
VECTORIZER_MIN_DF = 20
MLE_GRID_SIZE = 501
LAMBDA_PENALTY = 10.0
MIN_TRAIN_DOCS = 5000
BATCH_SIZE = 5000

TOKEN_COL = "body_200_tokens"
LABEL_COL = "ret_3day_cum"

def _as_tokens(x):
    return x if isinstance(x, list) else []

def _normalized_rank(values):
    return pd.Series(values).rank(method="first").to_numpy(dtype=np.float64) / len(values)

if "source_year" in df.columns:
    year_s = pd.to_numeric(df["source_year"], errors="coerce").astype("Int64")
else:
    year_s = pd.to_datetime(df["source_date"].astype(str), format="%Y%m%d", errors="coerce").dt.year.astype("Int64")

label_s = pd.to_numeric(df[LABEL_COL], errors="coerce")
eligible_train = ~df.get("is_duplicate", pd.Series(False, index=df.index)).fillna(False).astype(bool)

token_docs = df[TOKEN_COL].map(_as_tokens)

vectorizer = CountVectorizer(
    analyzer=lambda x: x,
    lowercase=False,
    min_df=VECTORIZER_MIN_DF,
    dtype=np.float32,
)
X_all = vectorizer.fit_transform(token_docs)

years = sorted(int(y) for y in year_s.dropna().unique())
test_years = [y for y in years if y - TRAIN_YEARS >= min(years)]

SESTM_sentiment = np.full(len(df), np.nan, dtype=np.float32)
model_rows = []
p_grid = np.linspace(1e-4, 1 - 1e-4, MLE_GRID_SIZE, dtype=np.float64)

for test_year in test_years:
    train_mask = (
        year_s.between(test_year - TRAIN_YEARS, test_year - 1)
        & label_s.notna()
        & eligible_train
    ).fillna(False).to_numpy(dtype=bool)

    test_mask = year_s.eq(test_year).fillna(False).to_numpy(dtype=bool)

    train_idx = np.flatnonzero(train_mask)
    test_idx = np.flatnonzero(test_mask)

    if len(train_idx) < MIN_TRAIN_DOCS or len(test_idx) == 0:
        print(f"{test_year}: skipped")
        continue

    y_train = label_s.iloc[train_idx].to_numpy(dtype=np.float64)
    X_train = X_all[train_idx]

    total_counts = np.asarray(X_train.sum(axis=0)).ravel()
    pos_counts = np.asarray(X_train[y_train > 0].sum(axis=0)).ravel()

    pi_hat = float((y_train > 0).mean())
    f = np.divide(
        pos_counts,
        total_counts,
        out=np.full_like(total_counts, np.nan),
        where=total_counts > 0,
    )

    valid_words = total_counts >= MIN_WORD_COUNT
    pos_score = f - pi_hat
    neg_score = pi_hat - f

    pos_candidates = np.flatnonzero(valid_words & np.isfinite(pos_score) & (pos_score > 0))
    neg_candidates = np.flatnonzero(valid_words & np.isfinite(neg_score) & (neg_score > 0))

    pos_sel = pos_candidates[np.argsort(pos_score[pos_candidates])[-N_POS_WORDS:]]
    neg_sel = neg_candidates[np.argsort(neg_score[neg_candidates])[-N_NEG_WORDS:]]

    selected = np.unique(np.r_[pos_sel, neg_sel])
    if len(selected) < 2:
        print(f"{test_year}: skipped, no selected words")
        continue

    Xs_train = X_train[:, selected].tocsr()
    s_train = np.asarray(Xs_train.sum(axis=1)).ravel()
    usable = s_train > 0

    if usable.sum() < MIN_TRAIN_DOCS:
        print(f"{test_year}: skipped, too few selected-word docs")
        continue

    Xs_use = Xs_train[usable]
    y_use = y_train[usable]

    p_rank = _normalized_rank(y_use)
    tau_hat = float(p_rank.mean())

    H = Xs_use.multiply(1.0 / np.asarray(Xs_use.sum(axis=1)).ravel()[:, None])
    W = np.column_stack([p_rank, 1.0 - p_rank])

    O_hat = np.asarray(H.T @ W) @ np.linalg.pinv(W.T @ W)
    O_hat = np.maximum(O_hat, 0)

    fallback = np.asarray(H.mean(axis=0)).ravel()
    fallback = np.maximum(fallback, 0)
    fallback = fallback / fallback.sum()

    for c in range(2):
        col_sum = O_hat[:, c].sum()
        O_hat[:, c] = fallback if col_sum <= 0 else O_hat[:, c] / col_sum

    prior = 2.0 * LAMBDA_PENALTY * (
        tau_hat * np.log(p_grid) + (1.0 - tau_hat) * np.log1p(-p_grid)
    )

    log_mix = np.log(
        np.maximum(
            p_grid[:, None] * O_hat[:, 0][None, :]
            + (1.0 - p_grid[:, None]) * O_hat[:, 1][None, :],
            1e-12,
        )
    )

    Xs_test = X_all[test_idx][:, selected].tocsr()
    s_test = np.asarray(Xs_test.sum(axis=1)).ravel()

    year_scores = np.full(len(test_idx), tau_hat, dtype=np.float32)
    active = np.flatnonzero(s_test > 0)

    for start in range(0, len(active), BATCH_SIZE):
        batch = active[start : start + BATCH_SIZE]
        ll = np.asarray(Xs_test[batch] @ log_mix.T, dtype=np.float64) + prior[None, :]
        year_scores[batch] = p_grid[np.argmax(ll, axis=1)].astype(np.float32)

    SESTM_sentiment[test_idx] = year_scores

    model_rows.append({
        "test_year": test_year,
        "train_start_year": test_year - TRAIN_YEARS,
        "train_end_year": test_year - 1,
        "train_docs": int(len(train_idx)),
        "topic_docs": int(usable.sum()),
        "test_docs": int(len(test_idx)),
        "scored_docs": int(np.isfinite(year_scores).sum()),
        "pi_hat": pi_hat,
        "tau_hat": tau_hat,
        "selected_words": int(len(selected)),
        "positive_words": int(len(pos_sel)),
        "negative_words": int(len(neg_sel)),
    })

    print(f"{test_year}: scored {len(test_idx):,} rows, selected_words={len(selected)}")

df["SESTM_sentiment"] = SESTM_sentiment
sestm_model_summary = pd.DataFrame(model_rows)

display(sestm_model_summary)
df[["source_date", "gicode", "company_name", "ret_3day_cum", "SESTM_sentiment"]].head()

In [ ]:
df[["source_date", "gicode", "company_name", "ret_3day_cum", "SESTM_sentiment"]].tail()

---
### **NB**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

NB_TRAIN_YEARS = 5
NB_MIN_DF = 50
NB_ALPHA = 1.0
NB_MIN_TRAIN_DOCS = 5000

TOKEN_COL = "body_200_tokens"
LABEL_COL = "ret_3day_cum"

def _as_tokens(x):
    return x if isinstance(x, list) else []

if "source_year" in df.columns:
    year_s = pd.to_numeric(df["source_year"], errors="coerce").astype("Int64")
else:
    year_s = pd.to_datetime(
        df["source_date"].astype(str),
        format="%Y%m%d",
        errors="coerce",
    ).dt.year.astype("Int64")

label_s = pd.to_numeric(df[LABEL_COL], errors="coerce")
eligible_train = ~df.get("is_duplicate", pd.Series(False, index=df.index)).fillna(False).astype(bool)

years = sorted(int(y) for y in year_s.dropna().unique())
test_years = [y for y in years if y - NB_TRAIN_YEARS >= min(years)]

NB_sentiment = np.full(len(df), np.nan, dtype=np.float32)
nb_model_rows = []

for test_year in test_years:
    train_mask = (
        year_s.between(test_year - NB_TRAIN_YEARS, test_year - 1)
        & label_s.notna()
        & eligible_train
    ).fillna(False).to_numpy(dtype=bool)

    test_mask = year_s.eq(test_year).fillna(False).to_numpy(dtype=bool)

    train_idx = np.flatnonzero(train_mask)
    test_idx = np.flatnonzero(test_mask)

    if len(train_idx) < NB_MIN_TRAIN_DOCS or len(test_idx) == 0:
        print(f"{test_year}: skipped")
        continue

    y_train = (label_s.iloc[train_idx].to_numpy(dtype=np.float64) > 0).astype(np.int8)

    if len(np.unique(y_train)) < 2:
        print(f"{test_year}: skipped, only one class")
        continue

    vectorizer = CountVectorizer(
        analyzer=lambda x: x,
        lowercase=False,
        min_df=NB_MIN_DF,
        dtype=np.int32,
    )

    try:
        X_train = vectorizer.fit_transform(df.iloc[train_idx][TOKEN_COL].map(_as_tokens))
    except ValueError:
        print(f"{test_year}: skipped, empty vocabulary")
        continue

    X_test = vectorizer.transform(df.iloc[test_idx][TOKEN_COL].map(_as_tokens))

    nb_model = MultinomialNB(alpha=NB_ALPHA)
    nb_model.fit(X_train, y_train)

    pos_col = int(np.flatnonzero(nb_model.classes_ == 1)[0])
    year_scores = nb_model.predict_proba(X_test)[:, pos_col].astype(np.float32)

    NB_sentiment[test_idx] = year_scores

    nb_model_rows.append({
        "test_year": test_year,
        "train_start_year": test_year - NB_TRAIN_YEARS,
        "train_end_year": test_year - 1,
        "train_docs": int(len(train_idx)),
        "test_docs": int(len(test_idx)),
        "scored_docs": int(np.isfinite(year_scores).sum()),
        "positive_label_rate": float(y_train.mean()),
        "vocab_size": int(len(vectorizer.get_feature_names_out())),
        "alpha": float(NB_ALPHA),
        "min_df": int(NB_MIN_DF),
    })

    print(
        f"{test_year}: scored {len(test_idx):,} rows, "
        f"vocab={len(vectorizer.get_feature_names_out()):,}, "
        f"positive_label_rate={y_train.mean():.3f}"
    )

df["NB_sentiment"] = NB_sentiment
nb_model_summary = pd.DataFrame(nb_model_rows)

display(nb_model_summary)
df[["source_date", "gicode", "company_name", "ret_3day_cum", "SESTM_sentiment", "NB_sentiment"]].head()

In [ ]:
df[["source_date", "gicode", "company_name", "ret_3day_cum", "SESTM_sentiment", "NB_sentiment"]].tail()

---
### **sentiment metrics**

In [ ]:
price_df = pd.read_parquet("../data/daily_adjusted_close.parquet")

price_gicodes = price_df["gicode"].astype(str).drop_duplicates().tolist()

date_cols = [c for c in price_df.columns if c not in ["gicode", "company_name"]]
trading_dates = pd.to_datetime(date_cols).sort_values()
trading_dates_np = trading_dates.to_numpy(dtype="datetime64[D]")

tmp = df[["source_date", "gicode", "SESTM_sentiment", "NB_sentiment"]].copy()
tmp["gicode"] = tmp["gicode"].astype(str)
tmp["source_dt"] = pd.to_datetime(tmp["source_date"].astype(str), format="%Y%m%d", errors="coerce")

# 비거래일 기사는 다음 거래일의 sentiment signal로 배정
source_np = tmp["source_dt"].to_numpy(dtype="datetime64[D]")
signal_idx = np.searchsorted(trading_dates_np, source_np, side="left")

valid = (
    tmp["source_dt"].notna().to_numpy()
    & (signal_idx >= 0)
    & (signal_idx < len(trading_dates_np))
    & tmp["gicode"].isin(price_gicodes).to_numpy()
)

tmp = tmp.loc[valid].copy()
tmp["signal_date"] = trading_dates_np[signal_idx[valid]]
tmp["signal_date"] = pd.to_datetime(tmp["signal_date"]).dt.strftime("%Y-%m-%d")

index_dates = trading_dates.strftime("%Y-%m-%d")

SESTM_sentiment_wide = (
    tmp.pivot_table(
        index="signal_date",
        columns="gicode",
        values="SESTM_sentiment",
        aggfunc="mean",
    )
    .reindex(index=index_dates, columns=price_gicodes)
    .astype("float32")
)

NB_sentiment_wide = (
    tmp.pivot_table(
        index="signal_date",
        columns="gicode",
        values="NB_sentiment",
        aggfunc="mean",
    )
    .reindex(index=index_dates, columns=price_gicodes)
    .astype("float32")
)

SESTM_sentiment_wide.index.name = "date"
NB_sentiment_wide.index.name = "date"

In [ ]:
price_wide = price_df.set_index("gicode").drop(columns=["company_name"]).T.rename_axis("date")
price_wide

In [ ]:
open_wide = pd.read_parquet("adjusted_open_wide.parquet")

In [ ]:
SESTM_sentiment_wide.to_parquet("SESTM_sentiment_wide.parquet")
NB_sentiment_wide.to_parquet("NB_sentiment_wide.parquet")
price_wide.to_parquet("price_wide.parquet")

df.to_pickle("news_tokenized_labeled_scored.pkl")